In [2]:
import pandas as pd
import numpy as np
import sklearn

df = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")
print("sklearn", sklearn.__version__, "| appointments:", df.shape, "| clinics:", clinics.shape)

sklearn 1.9.0 | appointments: (990, 16) | clinics: (8, 7)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("appointments_clean.csv")     

features = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
            "reminder_sent", "booking_channel", "visit_type"]
X = df[features].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})    
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)        
y = df["no_show"]                                                         
print("X columns:", X.shape[1])
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)                            

print("Train:", len(X_train), "| Test:", len(X_test))
print("No-shows in test:", y_test.sum())                             
baseline = (y_test == 0).mean()                                         
print("Baseline accuracy:", round(baseline, 4))

X columns: 11
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train: 792 | Test: 198
No-shows in test: 49
Baseline accuracy: 0.7525


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)     
X_test_s = scaler.transform(X_test)      

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)             

pred = model.predict(X_test_s)
prob = model.predict_proba(X_test_s)[:, 1]    

print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Confusion:\n", confusion_matrix(y_test, pred))
print("Precision:", round(precision_score(y_test, pred), 2))
print("Recall   :", round(recall_score(y_test, pred), 2))
print("F1       :", round(f1_score(y_test, pred), 2))
print("ROC-AUC  :", round(roc_auc_score(y_test, prob), 3))  

coefs = pd.Series(model.coef_[0], index=X.columns)
print(coefs.round(2))

i = list(X.columns).index("reminder_sent")
c = model.coef_[0][i]
print("exp(coef)            :", round(np.exp(c), 2))
print("exp(coef / scale)    :", round(np.exp(c / scaler.scale_[i]), 2))  

Accuracy : 0.788
Confusion:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall   : 0.35
F1       : 0.45
ROC-AUC  : 0.789
age                       -0.25
distance_km                0.25
fee_inr                    0.45
lead_days                  0.61
prior_no_shows             0.65
reminder_sent             -0.58
booking_channel_App       -0.05
booking_channel_Phone      0.22
booking_channel_Walk-In   -0.19
visit_type_Consultation    0.02
visit_type_Follow-up      -0.02
dtype: float64
exp(coef)            : 0.56
exp(coef / scale)    : 0.31


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# --- Full-depth tree ---
tree_full = DecisionTreeClassifier(random_state=42)             
tree_full.fit(X_train, y_train)                                      
print("Full tree depth:", tree_full.get_depth())
print("  Train acc:", round(tree_full.score(X_train, y_train), 3))
print("  Test acc :", round(tree_full.score(X_test, y_test), 3))   

# --- Depth-3 tree ---
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)   
tree3.fit(X_train, y_train)
print("Depth-3 tree")
print("  Train acc:", round(tree3.score(X_train, y_train), 3))
print("  Test acc :", round(tree3.score(X_test, y_test), 3))

Full tree depth: 15
  Train acc: 1.0
  Test acc : 0.672
Depth-3 tree
  Train acc: 0.822
  Test acc : 0.778


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]                      
print("RF train acc:", round(rf.score(X_train, y_train), 3))
print("RF test acc :", round(accuracy_score(y_test, rf_pred), 3))
print("RF recall   :", round(recall_score(y_test, rf_pred), 2))    
print("RF ROC-AUC  :", round(roc_auc_score(y_test, rf_prob), 3))     

imp = pd.Series(rf.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False).head(3))             

# --- Comparison table ---
t3_pred = tree3.predict(X_test)
t3_prob = tree3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "model":    ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, pred), accuracy_score(y_test, t3_pred), accuracy_score(y_test, rf_pred)],
    "recall":   [recall_score(y_test, pred),   recall_score(y_test, t3_pred),   recall_score(y_test, rf_pred)],
    "roc_auc":  [roc_auc_score(y_test, prob),  roc_auc_score(y_test, t3_prob),   roc_auc_score(y_test, rf_prob)],  # BLANK 10
}).round(3)
print(comparison)

RF train acc: 1.0
RF test acc : 0.747
RF recall   : 0.31
RF ROC-AUC  : 0.706
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64
                     model  accuracy  recall  roc_auc
0      Logistic regression     0.788   0.347    0.789
1  Decision tree (depth 3)     0.778   0.224    0.700
2            Random forest     0.747   0.306    0.706


In [11]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# ---------- (a) Tune the forest ----------
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")                           
grid.fit(X_train, y_train)                                                  
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

# ---------- (b) Choose the model (training data only) ----------
lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(), 
    "Random forest": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                     X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned forest": grid.best_score_,                                       
}
for name, score in cv_scores.items():
    print(f"{name:25s} CV ROC-AUC = {score:.3f}")



Best params: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.767
Logistic regression       CV ROC-AUC = 0.774
Decision tree (depth 3)   CV ROC-AUC = 0.678
Random forest             CV ROC-AUC = 0.747
Tuned forest              CV ROC-AUC = 0.767


In [12]:
# ---------- (c) Choose the threshold (training data only) ----------
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for t in [0.5, 0.35]:
    flags = (oof >= t).astype(int)                                   
    print(f"t={t}: precision={precision_score(y_train, flags):.2f}  "
          f"recall={recall_score(y_train, flags):.2f}  "
          f"F1={f1_score(y_train, flags):.2f}  flagged={flags.sum()}") 



t=0.5: precision=0.58  recall=0.25  F1=0.35  flagged=73
t=0.35: precision=0.51  recall=0.47  F1=0.49  flagged=154


In [ ]:
THRESHOLD = 0.35                                                    

# ---------- (d) Fit on train, test ONCE ----------
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]                                
flag = (probs >= THRESHOLD).astype(int)
print("Test precision:", round(precision_score(y_test, flag), 2))
print("Test recall   :", round(recall_score(y_test, flag), 2))
print("Flagged       :", flag.sum())
print("Test ROC-AUC  :", round(roc_auc_score(y_test, probs), 3))         



Test precision: 0.57
Test recall   : 0.57
Flagged       : 49
Test ROC-AUC  : 0.789


In [ ]:
# ---------- (e) Save the risk scores ----------
risk = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,             
    "no_show": y_test.values,
    "risk_prob": probs.round(4),
    "flag": flag,
})
risk.to_csv("test_risk_scores.csv", index=False)
print(risk.shape, "| flagged:", risk["flag"].sum())

(198, 4) | flagged: 49
